# Testing Stepfun GOT-OCR-2.0-hf

This notebook tests the GOT-OCR-2.0-hf model on Apple Silicon M2.

**Goals**
- Verify MPS (Metal Performance Shaders) availability
- Load GOT-OCR-2.0-hf model from HuggingFace
- Benchmark MPS vs CPU performance
- Test inference with sample images
- Measure loading time, inference time, and memory usage

## Setup and Imports

In [ ]:
import time

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

## Load Model and Processor

Load the GOT-OCR-2.0-hf model and measure loading time.

In [ ]:
MODEL_ID = "stepfun-ai/GOT-OCR-2.0-hf"

print(f"Loading model: {MODEL_ID}")
print("This may take a few minutes on first run (downloading ~580MB)...\n")

# Measure loading time
start_time = time.time()

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID)
    loading_time = time.time() - start_time
    print(f"✅ Model loaded successfully in {loading_time:.2f} seconds")

except Exception as e:
    print(f"❌ Error loading model: {e}")
    raise

Loading model: stepfun-ai/GOT-OCR-2.0-hf
This may take a few minutes on first run (downloading ~580MB)...



'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: bad21bd4-b07f-4d60-985a-10861ea87849)')' thrown while requesting HEAD https://huggingface.co/stepfun-ai/GOT-OCR-2.0-hf/resolve/main/processor_config.json
Retrying in 1s [Retry 1/5].


✅ Model loaded successfully in 26.59 seconds


## Test Inference with Sample Image on CPU

In [ ]:
# Run inference
test_image_url = "https://huggingface.co/datasets/hf-internal-testing/fixtures_got_ocr/resolve/main/image_ocr.jpg"
print("Running inference on CPU...")

start_time = time.time()

try:
    # Process image
    inputs = processor(test_image_url, return_tensors="pt").to("cpu")

    # Generate output
    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=4096,
        )

    # Decode output
    result = processor.decode(
        generate_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )

    inference_time = time.time() - start_time

    print(f"\n✅ Inference completed in {inference_time:.2f} seconds")
    print(f"\nOCR Result:\n{result}")

except Exception as e:
    print(f"❌ Error during inference: {e}")
    import traceback

    traceback.print_exc()

Running inference on CPU...

✅ Inference completed in 78.52 seconds

OCR Result:
R&D QUALITY IMPROVEMENT
SUGGESTION/SOLUTION FORM
Name/Phone Ext. : M. Hamann, P. Harper, P. Martinez
Date: 9/3/92
Supervisor/Manager: J. S. Wigand
R&D Group: Licensee
Suggestion:
Discontinue coal retention analyses on licensee submitted
product samples. (Note: Coal Retention testing is not
performed by most licensees. Other B&W physical
measurements as ends stability and inspection for soft
spots in cigarettes are thought to be sufficient measures
to assure cigarette physical integrity. The proposed
action will increase laboratory productivity.)
Suggested Solution(s) : Delete coal retention from the list of standard
analyses performed on licensee submitted
product samples. Special requests for coal
retention testing could still be submitted on
an exception basis.
Have you contacted your Manager/Supervisor?
Yes
No
Manager Comments: Manager, please contact suggester and forward
comments to the Quality Counci

## Test Inference with Sample Image on MPS

In [ ]:
print("moving the model to mps")
model.to("mps")
start_time = time.time()

try:
    # Process image
    print("processing image")
    inputs = processor(test_image_url, return_tensors="pt").to("mps")

    # Generate output
    print("inference")
    with torch.no_grad():
        generate_ids = model.generate(
            **inputs,
            do_sample=False,
            tokenizer=processor.tokenizer,
            stop_strings="<|im_end|>",
            max_new_tokens=4096,
        )

    # Decode output
    print("process model outputs")
    result = processor.decode(
        generate_ids[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )

    inference_time = time.time() - start_time

    print(f"\n✅ Inference completed in {inference_time:.2f} seconds")
    print(f"\nOCR Result:\n{result}")

except Exception as e:
    print(f"❌ Error during inference: {e}")
    import traceback

    traceback.print_exc()

moving the model to mps
processing image
inference
process model outputs

✅ Inference completed in 222.12 seconds

OCR Result:
R&D QUALITY IMPROVEMENT
SUGGESTION/SOLUTION FORM
Name/Phone Ext. : M. Hamann, P. Harper, P. Martinez
Date: 9/3/92
Supervisor/Manager: J. S. Wigand
R&D Group: Licensee
Suggestion:
Discontinue coal retention analyses on licensee submitted
product samples. (Note: Coal Retention testing is not
performed by most licensees. Other B&W physical
measurements as ends stability and inspection for soft
spots in cigarettes are thought to be sufficient measures
to assure cigarette physical integrity. The proposed
action will increase laboratory productivity.)
Suggested Solution(s) : Delete coal retention from the list of standard
analyses performed on licensee submitted
product samples. Special requests for coal
retention testing could still be submitted on
an exception basis.
Have you contacted your Manager/Supervisor?
Yes
No
Manager Comments: Manager, please contact sugges

- 1.10m to move the model to mps and a few seconds to process image
- 222.12 seconds for a complete inference

## Summary & Findings

### Test Results:
- ✅ Model: GOT-OCR-2.0-hf loads successfully on M2
- ✅ Model loading time: ~26 seconds
- ✅ CPU inference time: ~80 seconds per image
- ✅ OCR quality: Excellent (successfully extracted complex document text)
- ⚠️ **MPS performance issue discovered**: MPS is SLOWER than CPU (~222s vs ~80s)

### Key Decision:
**Proceeding with CPU-only deployment**
- MPS acceleration provides no benefit for this model
- 80 seconds per image is acceptable for low-volume, occasional use
- Simpler architecture without MPS complexity